# Multi-modal 2-D UNet++ Patient-level Predictor & Interactive Visualizer

This notebook allows you to select a trained UNet++ model checkpoint (`.h5` file) and a specific test patient to:
1. Predict the full 3D lesion segmentation mask slice-by-slice.
2. Symmetrically crop or pad the slices back to their original input shape.
3. Save the predicted mask as a NIfTI volume (`pred_mask.nii.gz`) inside the results folder.
4. Save the final patient-level metrics (Dice, IoU, confusion matrix) as `prediction_details.json` inside the results folder.
5. Interactively visualize all modalities (T1, T2, FLAIR), the Ground Truth mask, and the Predicted mask slice-by-slice with a single slider.

In [7]:
import os
os.environ["SM_FRAMEWORK"] = "tf.keras"
import sys
import json
import numpy as np
import nibabel as nib
import tensorflow as tf
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
import segmentation_models as sm

# Ensure path imports from src
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from dataset import load_volume, extract_slices
from model import UNetPlusPlus

### 1. Configuration
Specify the trained model checkpoint path and the patient ID in the test dataset.

In [14]:
# Path to the model checkpoint and test subject
model_path = "/home/darshan/MS/models/unetpp/runs/adam_0.001_8_1/best_model.h5"
subject_id = "S177"  # Choose a subject from the test split (e.g. S7, S47, S152, S149, S12, S105)

### 2. Run Predictions & Save NIfTI + JSON Results

In [16]:
def centre_pad_crop_2d(arr, target_h, target_w):
    h, w = arr.shape
    if h > target_h:
        sh = (h - target_h) // 2
        arr = arr[sh:sh + target_h, ...]
        h = target_h
    if w > target_w:
        sw = (w - target_w) // 2
        arr = arr[:, sw:sw + target_w]
        w = target_w
    pad_h = target_h - h
    pad_w = target_w - w
    if pad_h > 0 or pad_w > 0:
        top = pad_h // 2
        bottom = pad_h - top
        left = pad_w // 2
        right = pad_w - left
        arr = np.pad(arr, ((top, bottom), (left, right)), mode='constant', constant_values=0)
    return arr

# Verify paths
script_dir = os.getcwd()
preprocessed_dir = os.path.abspath(os.path.join(script_dir, "..", "..", "..", "data", "PREPROCESSED256"))
subj_dir = os.path.join(preprocessed_dir, subject_id)

t1_path = os.path.join(subj_dir, "t1.nii.gz")
t2_path = os.path.join(subj_dir, "t2.nii.gz")
flair_path = os.path.join(subj_dir, "flair.nii.gz")
mask_path = os.path.join(subj_dir, "mask.nii.gz")

if not all(os.path.exists(p) for p in [t1_path, t2_path, flair_path, mask_path]):
    raise FileNotFoundError(f"One or more NIfTI files missing for subject {subject_id} in {subj_dir}")

# Extract model run name to determine results folder
run_name = os.path.basename(os.path.dirname(model_path))
results_dir = os.path.abspath(os.path.join(script_dir, "..", "results", run_name))
os.makedirs(results_dir, exist_ok=True)

print(f"Results will be saved to: {results_dir}")

# 1. Load original volume data (unpadded)
t1_img = nib.load(t1_path)
t1_data = t1_img.get_fdata(dtype=np.float32)
t2_data = nib.load(t2_path).get_fdata(dtype=np.float32)
flair_data = nib.load(flair_path).get_fdata(dtype=np.float32)
mask_data = (nib.load(mask_path).get_fdata(dtype=np.float32) > 0.5).astype(np.float32)

H_orig, W_orig, D_orig = flair_data.shape
print(f"Original volume dimensions: Height={H_orig}, Width={W_orig}, Depth={D_orig}")

# 2. Load model
with tf.device('/GPU:1'):
    # Load your model and run predictions inside this block
    # model = keras.models.load_model('my_model.keras')
    # predictions = model.predict(data)
    print("Loading trained UNet++ model checkpoint...")
    class DummyDiceScore(tf.keras.metrics.Metric):
        def __init__(self, **kwargs): super().__init__(name='dice_score')
    class DummyIoUScore(tf.keras.metrics.Metric):
        def __init__(self, **kwargs): super().__init__(name='iou_score')

    model = tf.keras.models.load_model(
        model_path,
        custom_objects={
            'dice_loss': sm.losses.dice_loss,
            'dice_score': DummyDiceScore,
            'iou_score': DummyIoUScore
        },
        compile=False
    )

# 3. Extract prepared slices for prediction 
    print("Preparing modalities for model prediction...")
    imgs_prep, masks_prep = extract_slices(t1_path, t2_path, flair_path, mask_path, is_train=False, skip_blank_ratio=0.0)

    # Predict slice-by-slice
    print("Running model inference...")
    preds = model.predict(imgs_prep, batch_size=4, verbose=1)
    preds_bin = (preds > 0.5).astype(np.float32)

# 4. Reconstruct 3D Predicted Mask (pad/crop back to original size)
pred_mask_3d = np.zeros_like(mask_data)
for z in range(D_orig):
    pred_slice_2d = preds_bin[z, :, :, 0]
    # Slices are already 256x256
    pred_slice_orig = centre_pad_crop_2d(pred_slice_2d, mask_data.shape[0], mask_data.shape[1])
    pred_mask_3d[:, :, z] = pred_slice_orig

# 5. Save Predicted Mask as NIfTI
pred_nii = nib.Nifti1Image(pred_mask_3d, t1_img.affine, t1_img.header)
pred_mask_path = os.path.join(results_dir, "pred_mask.nii.gz")
nib.save(pred_nii, pred_mask_path)
print(f"Saved predicted mask volume: {pred_mask_path}")

# 6. Compute metrics
inter = np.sum((mask_data == 1.0) & (pred_mask_3d == 1.0))
union_dice = np.sum(mask_data == 1.0) + np.sum(pred_mask_3d == 1.0)
union_iou = np.sum((mask_data == 1.0) | (pred_mask_3d == 1.0))

dice = float((2.0 * inter) / (union_dice + 1e-7)) if union_dice > 0 else 1.0
iou = float(inter / (union_iou + 1e-7)) if union_iou > 0 else 1.0

tp = int(np.sum((mask_data == 1.0) & (pred_mask_3d == 1.0)))
tn = int(np.sum((mask_data == 0.0) & (pred_mask_3d == 0.0)))
fp = int(np.sum((mask_data == 0.0) & (pred_mask_3d == 1.0)))
fn = int(np.sum((mask_data == 1.0) & (pred_mask_3d == 0.0)))

metrics = {
    "subject_id": subject_id,
    "dice": round(dice, 4),
    "iou": round(iou, 4),
    "confusion_matrix": {
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn
    }
}

# Save prediction details JSON
details_path = os.path.join(results_dir, "prediction_details.json")
with open(details_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved final metrics: {details_path}")
print(json.dumps(metrics, indent=2))

Results will be saved to: /home/darshan/MS/models/unetpp/results/adam_0.001_8_1
Original volume dimensions: Height=256, Width=256, Depth=182
Loading trained UNet++ model checkpoint...
Preparing modalities for model prediction...
Running model inference...
46/46 [==============================] - 2s 40ms/step
Saved predicted mask volume: /home/darshan/MS/models/unetpp/results/adam_0.001_8_1/pred_mask.nii.gz
Saved final metrics: /home/darshan/MS/models/unetpp/results/adam_0.001_8_1/prediction_details.json
{
  "subject_id": "S177",
  "dice": 0.5078,
  "iou": 0.3403,
  "confusion_matrix": {
    "TP": 2672,
    "TN": 11919701,
    "FP": 1646,
    "FN": 3533
  }
}


### 3. Interactive Visualization
Use the slice slider below to dynamically scroll through the entire 3D volume. You will see T1, T2, FLAIR, Ground Truth, and Predicted Mask perfectly aligned!

In [17]:
def plot_slice(z):
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    
    # Rotate 90 degrees to align orientation
    s_t1 = np.rot90(imgs_prep[z, :, :, 0])
    s_t2 = np.rot90(imgs_prep[z, :, :, 1])
    s_fl = np.rot90(imgs_prep[z, :, :, 2])
    s_gt = np.rot90(masks_prep[z, :, :, 0])
    s_pr = np.rot90(preds_bin[z, :, :, 0])
    
    axes[0].imshow(s_t1, cmap='gray')
    axes[0].set_title(f'T1w (z={z})')
    axes[0].axis('off')
    
    axes[1].imshow(s_t2, cmap='gray')
    axes[1].set_title(f'T2w (z={z})')
    axes[1].axis('off')
    
    axes[2].imshow(s_fl, cmap='gray')
    axes[2].set_title(f'FLAIR (z={z})')
    axes[2].axis('off')
    
    axes[3].imshow(s_gt, cmap='gray')
    axes[3].set_title('Ground Truth')
    axes[3].axis('off')
    
    axes[4].imshow(s_pr > 0.5, cmap='gray')
    axes[4].set_title('Predicted Mask')
    axes[4].axis('off')
    
    plt.tight_layout()
    plt.show()

interact(plot_slice, z=widgets.IntSlider(min=0, max=D_orig-1, step=1, value=D_orig//2, description='Slice'))

interactive(children=(IntSlider(value=91, description='Slice', max=181), Output()), _dom_classes=('widget-inte…

<function __main__.plot_slice(z)>